# Contract Workflow HF API Server

Notebook này load model **[d90nqm/contract-workflow](https://huggingface.co/d90nqm/contract-workflow)** (DebertaV2ForSequenceClassification, 8 classes) trực tiếp từ Hugging Face Hub.

## Luồng xử lý
1. **Nhận** `contract_text` + `clause_types` từ `workflow_service` qua HTTP POST
2. **Phân loại** loại hợp đồng: `WF_NDA`, `WF_SERVICE`, `WF_EMPLOYMENT`, `WF_EXECUTIVE`, `WF_VENDOR`, `WF_PURCHASE`, `WF_PROCUREMENT`, `WF_GENERAL`
3. **Tra bảng** step matrix → danh sách các bước duyệt theo thứ tự + `role_id` người được ký duyệt
4. **Trả về** JSON chuẩn cho `workflow_service` để tạo Workflow trong DB

## Sau khi chạy
Copy URL Ngrok in ra và set vào biến môi trường `KAGGLE_AI_URL` trên server Django.


In [ ]:
# Cell 1: Install dependencies
!pip install -q fastapi uvicorn pyngrok nest_asyncio transformers torch sentencepiece protobuf accelerate


In [ ]:
# Cell 2: Imports & FastAPI app
import os, re, unicodedata, torch, nest_asyncio, uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nest_asyncio.apply()
app = FastAPI(title='Contract Workflow HF API Server', version='1.0')

# ── Load model from HuggingFace Hub ──────────────────────────────────────────
HF_MODEL_ID = 'd90nqm/contract-workflow'
print(f'Loading {HF_MODEL_ID} ...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_ID, torch_dtype=torch.float16 if device=='cuda' else torch.float32)
model.to(device).eval()
print(f'Model loaded on {device}. Labels: {list(model.config.id2label.values())}')

# ── Step matrix: WF_TYPE -> [ (step_name, role_id, description_vi) ] ─────────
# role_id mapping (from your system):
#   4=Legal/Signing  5=Manager  6=Finance  7=Technical  8=Security
#   9=Compliance  10=Procurement  11=Director/Executive
STEP_MATRIX = {
    'WF_NDA': [
        ('Legal Review',      4,  'Bộ phận pháp lý xem xét tính hợp lệ của thỏa thuận bảo mật.'),
        ('Manager Approval',  5,  'Quản lý trực tiếp phê duyệt chủ trương ký kết NDA.'),
        ('Contract Signing',  4,  'Đại diện có thẩm quyền ký kết thỏa thuận bảo mật.'),
        ('Document Archive',  4,  'Lưu trữ NDA đã ký vào hệ thống quản lý hợp đồng.'),
    ],
    'WF_EMPLOYMENT': [
        ('Legal Review',      4,  'Pháp lý kiểm tra điều khoản lao động, thử việc và chấm dứt hợp đồng.'),
        ('Finance Review',    6,  'Tài chính xác nhận ngân sách lương, phụ cấp và phúc lợi.'),
        ('Manager Approval',  5,  'Quản lý phê duyệt vị trí tuyển dụng và điều kiện hợp đồng.'),
        ('Contract Signing',  4,  'Ký kết hợp đồng lao động giữa người lao động và người sử dụng lao động.'),
        ('Document Archive',  4,  'Lưu hồ sơ lao động vào hệ thống nhân sự.'),
    ],
    'WF_EXECUTIVE': [
        ('Legal Review',      4,  'Pháp lý rà soát điều khoản bồi thường, thay đổi kiểm soát và cam kết đặc biệt.'),
        ('Finance Review',    6,  'Tài chính đánh giá gói thù lao, cổ phần và các nghĩa vụ tài chính dài hạn.'),
        ('Compliance Review', 9,  'Kiểm tra tuân thủ quy định quản trị doanh nghiệp và xung đột lợi ích.'),
        ('Director Approval', 11, 'Hội đồng quản trị hoặc Giám đốc cấp cao phê duyệt hợp đồng điều hành.'),
        ('Executive Approval',11, 'Phê duyệt tối cao từ Ban điều hành / Chủ tịch HĐQT.'),
        ('Contract Signing',  4,  'Ký kết hợp đồng điều hành có hiệu lực pháp lý.'),
        ('Document Archive',  4,  'Lưu trữ hợp đồng điều hành theo yêu cầu quản trị.'),
    ],
    'WF_SERVICE': [
        ('Technical Review',  7,  'Kỹ thuật thẩm định phạm vi dịch vụ, SLA và năng lực nhà cung cấp.'),
        ('Legal Review',      4,  'Pháp lý kiểm tra điều khoản trách nhiệm, sở hữu trí tuệ và bảo mật.'),
        ('Finance Review',    6,  'Tài chính xác nhận giá dịch vụ và điều kiện thanh toán.'),
        ('Manager Approval',  5,  'Quản lý phê duyệt phạm vi và chi phí dịch vụ.'),
        ('Contract Signing',  4,  'Đại diện ký kết hợp đồng dịch vụ.'),
        ('Document Archive',  4,  'Lưu hợp đồng dịch vụ vào kho lưu trữ.'),
    ],
    'WF_VENDOR': [
        ('Procurement Review',10, 'Mua sắm đánh giá năng lực, giá cả và điều kiện cung cấp của nhà cung cấp.'),
        ('Technical Review',  7,  'Kỹ thuật kiểm tra thông số kỹ thuật và khả năng tích hợp.'),
        ('Legal Review',      4,  'Pháp lý rà soát điều khoản bảo hành, phạt vi phạm và chấm dứt.'),
        ('Finance Review',    6,  'Tài chính xác nhận giá mua, chiết khấu và điều khoản thanh toán.'),
        ('Manager Approval',  5,  'Quản lý phê duyệt lựa chọn nhà cung cấp.'),
        ('Contract Signing',  4,  'Ký kết hợp đồng nhà cung cấp.'),
        ('Document Archive',  4,  'Lưu trữ hợp đồng nhà cung cấp.'),
    ],
    'WF_PURCHASE': [
        ('Finance Review',    6,  'Tài chính kiểm tra ngân sách và giá trị mua sắm.'),
        ('Procurement Review',10, 'Mua sắm xác nhận quy trình đấu thầu và lựa chọn nhà thầu.'),
        ('Legal Review',      4,  'Pháp lý rà soát điều khoản mua bán, chuyển nhượng tài sản.'),
        ('Manager Approval',  5,  'Quản lý phê duyệt quyết định mua sắm.'),
        ('Director Approval', 11, 'Giám đốc phê duyệt đối với các hợp đồng mua sắm có giá trị lớn.'),
        ('Contract Signing',  4,  'Ký kết hợp đồng mua bán.'),
        ('Document Archive',  4,  'Lưu trữ hợp đồng mua sắm.'),
    ],
    'WF_PROCUREMENT': [
        ('Procurement Review',10, 'Mua sắm lập kế hoạch, xây dựng yêu cầu và đánh giá nhà thầu.'),
        ('Finance Review',    6,  'Tài chính xem xét ngân sách khung và cam kết chi tiêu.'),
        ('Legal Review',      4,  'Pháp lý kiểm tra hợp đồng khung, điều khoản bảo hành và rủi ro pháp lý.'),
        ('Compliance Review', 9,  'Tuân thủ kiểm tra quy trình đấu thầu và quy định mua sắm công.'),
        ('Manager Approval',  5,  'Quản lý phê duyệt kế hoạch mua sắm.'),
        ('Director Approval', 11, 'Giám đốc phê duyệt hợp đồng mua sắm chiến lược.'),
        ('Contract Signing',  4,  'Ký kết hợp đồng mua sắm.'),
        ('Document Archive',  4,  'Lưu trữ toàn bộ hồ sơ mua sắm.'),
    ],
    'WF_GENERAL': [
        ('Legal Review',      4,  'Pháp lý rà soát tính hợp lệ và rủi ro tổng thể của hợp đồng.'),
        ('Manager Approval',  5,  'Quản lý xem xét và phê duyệt chủ trương ký kết.'),
        ('Contract Signing',  4,  'Ký kết hợp đồng chính thức.'),
        ('Document Archive',  4,  'Lưu trữ hợp đồng sau ký kết.'),
    ],
}

# ── Pydantic schemas ──────────────────────────────────────────────────────────
class RecommendWorkflowRequest(BaseModel):
    contract_text: str
    clause_types: List[str] = []
    contract_type: str = ''

class WorkflowStepResponse(BaseModel):
    step_name: str
    role_id: int
    description: str

class RecommendWorkflowResponse(BaseModel):
    workflow_type: str
    steps: List[WorkflowStepResponse]
    reasons: str
    workflow_name: Optional[str] = None

# ── Helper: clean text ────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ── Endpoint: health check ────────────────────────────────────────────────────
@app.get('/health')
def health():
    return {'status': 'healthy', 'model': 'd90nqm/contract-workflow', 'device': device}

# ── Endpoint: recommend workflow ──────────────────────────────────────────────
@app.post('/api/v1/recommend_workflow', response_model=RecommendWorkflowResponse)
async def recommend_workflow(payload: RecommendWorkflowRequest):
    text = clean_text(payload.contract_text)
    if not text:
        raise HTTPException(status_code=400, detail='contract_text is required')

    # Step 1: Classify workflow type using HF DeBERTa model
    try:
        enc = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=512,
            padding=True
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits

        pred_idx = int(torch.argmax(logits, dim=-1).item())
        workflow_type = model.config.id2label.get(pred_idx, 'WF_GENERAL')
        confidence = float(torch.softmax(logits, dim=-1)[0][pred_idx].item())
        print(f'[CLASSIFY] -> {workflow_type} (confidence={confidence:.2%})')

    except Exception as e:
        print(f'[CLASSIFY ERROR] {e}')
        workflow_type = 'WF_GENERAL'
        confidence = 0.0

    # Step 2: Map WF_TYPE -> steps with role_id & description
    step_defs = STEP_MATRIX.get(workflow_type, STEP_MATRIX['WF_GENERAL'])
    steps_list = [
        WorkflowStepResponse(step_name=name, role_id=role_id, description=desc)
        for name, role_id, desc in step_defs
    ]

    # Step 3: Build response
    clause_note = f" Detected clause signals: {', '.join(payload.clause_types[:5])}." if payload.clause_types else ''
    reasons = (
        f"Model d90nqm/contract-workflow classified this as '{workflow_type}' "
        f"with {confidence:.1%} confidence.{clause_note} "
        f"Workflow steps and approving roles were automatically mapped from the step matrix."
    )

    label_name = workflow_type.replace('WF_', '').replace('_', ' ').title()
    workflow_name = f'{label_name} Contract Approval Workflow'

    return RecommendWorkflowResponse(
        workflow_type=workflow_type,
        steps=steps_list,
        reasons=reasons,
        workflow_name=workflow_name
    )


In [ ]:
# Cell 3: Start Ngrok tunnel and Uvicorn server
# NOTE: Replace NGROK_TOKEN with your own token from https://dashboard.ngrok.com/
NGROK_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', 'YOUR_NGROK_TOKEN_HERE')
ngrok.set_auth_token(NGROK_TOKEN)

try:
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    print('\n' + '='*60)
    print('  NGROK PUBLIC URL (copy this to KAGGLE_AI_URL):')
    print(f'  {public_url}')
    print('  Endpoint: POST /api/v1/recommend_workflow')
    print('='*60 + '\n')
except Exception as e:
    print(f'Ngrok tunnel failed: {e}. Server will still run locally.')

uvicorn.run(app, host='0.0.0.0', port=8000)
